### 연습문제
- Doc2Vec 라이브러리 이용한 감정에 분석
- 데이터는 ratings_train.txt 파일을 로드
    - 특수 문자, 2칸 이상의 공백의 문자를 제거하는 정규화함수
    - document 컬럼의 데이터에서 중복 데이터를 제거
    - 빈 텍스트, " "가 존재한다면 해당 행 데이터도 제거
    - 상위의 5000개 정도 데이터를 이용
- 토큰화 함수 Komoran를 이용
    - 필요한 품사 : NNP, NNG, VV, VA, MAG, XR만을 사용
    - 불용어 단어 : 하다, 되다, 이다, 것, 수, 거 단어들은 제외
- 데이터에서 독립(document), 종속(label) 변수로 데이터를 나눠주고 train, test 데이터셋을 나눠준다 비율은 8:2
- Doc2Vec 객체를 생성하여 학습
    - 매개변수
        - vector_size = 200
        - window = 5
        - min_count = 2
        - dm = 1
        - negative = 5
        - seed = 42
        - epochs = 50
    - 학습시키는 데이터는 X_train
- X_train, X_test -> 문자열 데이터 -> infer_vector() 함수를 이용해서 임베딩
- 고전 머신러닝 분류 모델을 이용하여 임베딩된 데이터를 독립 변수로 Y의 데이터들을 종속 변수로 학습하여 예측
    - 정확도를 확인
    - LogisticRegression(max_iter=2000, random_state=42)
    - LinearSVC(random_state=42)
    - 두 개의 모델을 사용하여 정확도가 좋은 모델을 선택

In [139]:
import pandas as pd
from konlpy.tag import Komoran
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import re
import random
from sklearn.model_selection import train_test_split

In [163]:
data = pd.read_csv('../data/ratings_train.txt', sep='\t')
data

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1
...,...,...,...
149995,6222902,인간이 문제지.. 소는 뭔죄인가..,0
149996,8549745,평점이 너무 낮아서...,1
149997,9311800,이게 뭐요? 한국인은 거들먹거리고 필리핀 혼혈은 착하다?,0
149998,2376369,청춘 영화의 최고봉.방황과 우울했던 날들의 자화상,1


In [164]:
data['document']

0                                       아 더빙.. 진짜 짜증나네요 목소리
1                         흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나
2                                         너무재밓었다그래서보는것을추천한다
3                             교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정
4         사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...
                                ...                        
149995                                  인간이 문제지.. 소는 뭔죄인가..
149996                                        평점이 너무 낮아서...
149997                      이게 뭐요? 한국인은 거들먹거리고 필리핀 혼혈은 착하다?
149998                          청춘 영화의 최고봉.방황과 우울했던 날들의 자화상
149999                             한국 영화 최초로 수간하는 내용이 담긴 영화
Name: document, Length: 150000, dtype: object

In [165]:
data['document'] = [re.sub(r"[^가-힣0-9a-zA-Z\s\.]", " ", str(text)) for text in data['document']]

In [166]:
data['document'] = [re.sub(r"\s+", " ", text).strip() for text in data['document']]

In [167]:
# for text in data['document']:
#     text = re.sub(r"[^가-힣0-9a-zA-Z\s\.]", " ", str(text))
#     text = re.sub(r"\s+", " ", text).strip()

In [168]:
data['document'] = data['document'].str.strip()

In [169]:
data.drop_duplicates('document', inplace=True)
data

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화 스파이더맨에서 늙어보이기만 했던 커스틴 ...,1
...,...,...,...
149995,6222902,인간이 문제지.. 소는 뭔죄인가..,0
149996,8549745,평점이 너무 낮아서...,1
149997,9311800,이게 뭐요 한국인은 거들먹거리고 필리핀 혼혈은 착하다,0
149998,2376369,청춘 영화의 최고봉.방황과 우울했던 날들의 자화상,1


In [170]:
data.dropna(axis=0, inplace=True, ignore_index=True)

In [171]:
data.isna().sum(axis=0)

id          0
document    0
label       0
dtype: int64

In [172]:
data

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화 스파이더맨에서 늙어보이기만 했던 커스틴 ...,1
...,...,...,...
144730,6222902,인간이 문제지.. 소는 뭔죄인가..,0
144731,8549745,평점이 너무 낮아서...,1
144732,9311800,이게 뭐요 한국인은 거들먹거리고 필리핀 혼혈은 착하다,0
144733,2376369,청춘 영화의 최고봉.방황과 우울했던 날들의 자화상,1


In [173]:
df = data.head(5000)

In [174]:
# 형태소 분석 Komoran을 이용하여 토큰화
komoran = Komoran()

# 특정 품사만 사용
allow_pos = ['NNP', 'NNG', 'VV', 'VA', 'MAG', 'XR']
# 불용어
stop_word = ['하다', '되다', '이다', '것', '수', '거']


In [175]:
def tokenize(text):
    
    tokens = []
    for word, pos in komoran.pos(text):
        if word not in stop_word and len(word) > 1 and pos in allow_pos:
                tokens.append(word)
    return tokens

In [176]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        5000 non-null   int64 
 1   document  5000 non-null   object
 2   label     5000 non-null   int64 
dtypes: int64(2), object(1)
memory usage: 117.3+ KB


In [177]:
df

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화 스파이더맨에서 늙어보이기만 했던 커스틴 ...,1
...,...,...,...
4995,9246120,꼭 보세요. 꿀잼입니다,1
4996,10058146,재미도 없고 감동도 없고.. 평점 8.5 이해가 안감. 알바생들인가,0
4997,10077756,어디 디비디방에나 있을거같은 영화,0
4998,2887151,패러디 영화네 이거왠지 저질일듯,0


In [178]:
[tokenize(text) for text in df['document']]

[['더빙', '진짜', '짜증', '목소리'],
 ['포스터', '초딩', '영화', '오버', '연기', '가볍'],
 [],
 ['교도소', '이야기', '솔직히', '재미', '평점', '조정'],
 ['익살', '연기', '돋보이', '영화', '스파이더맨', '보이', '커스틴 던스트', '너무나'],
 ['걸음마', '초등학교', '학년', '영화', '반개', '아깝'],
 ['원작', '긴장감', '제대로', '살리'],
 ['반개',
  '아깝',
  '나오',
  '이응경',
  '길용우',
  '연기',
  '생활',
  '정말',
  '발로',
  '납치',
  '감금',
  '반복',
  '반복',
  '드라마',
  '가족',
  '연기',
  '못하',
  '사람',
  '모이'],
 ['액션', '재미', '영화'],
 ['평점', '헐리우드', '화려', '너무', '길들이'],
 [],
 ['눈물', '나서', '향수', '자극', '허진호', '감성', '절제', '멜로', '달인'],
 ['손들', '횡단보도', '건너', '뛰쳐나오', '이범수', '연기', '드럽'],
 ['담백', '깔끔', '신문', '기사', '로만', '보다', '자꾸', '잊어버리', '사람'],
 ['취향', '존중', '진짜', '극장', '영화', '가장', '감동', '스토리', '어거지', '감동', '어거지'],
 ['매번', '긴장'],
 ['사람', '웃기', '바스코', '이기', '락스', '바비', '이기', '아이돌', '그냥', '안달', '보이'],
 ['굿바이 레닌', '표절', '이해', '갈수록', '재미없'],
 ['이건', '정말', '깨알', '캐스팅', '질퍽', '산뜻', '내용', '구성', '깨알'],
 ['약탈', '위하', '변명', '착하', '절대'],
 ['심오', '그냥', '학생', '선생', '놀아나', '영화', '절대'],
 ['가능'],
 ['재미없',
  '지루',
  '음식',


In [179]:
df['document'] = [tokenize(text) for text in df['document']]

C:\Users\student\AppData\Local\Temp\ipykernel_7932\752334765.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['document'] = [tokenize(text) for text in df['document']]


In [180]:
df

,id,document,label
0,9976970,"[더빙, 진짜, 짜증, 목소리]",0
1,3819312,"[포스터, 초딩, 영화, 오버, 연기, 가볍]",1
2,10265843,[],0
3,9045019,"[교도소, 이야기, 솔직히, 재미, 평점, 조정]",0
4,6483659,"[익살, 연기, 돋보이, 영화, 스파이더맨, 보이, 커스틴 던스트, 너무나]",1
...,...,...,...
4995,9246120,[],1
4996,10058146,"[재미, 감동, 평점, 이해, 안감, 알바]",0
4997,10077756,"[비디, 영화]",0
4998,2887151,"[패러디 영화, 왠지, 저질]",0


In [181]:
X = df['document'].values
Y = df['label'].values

In [196]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)

In [197]:
tagged = []
for idx, toks in enumerate(X_train):
    # TaggedDocument를 이용하여 문장 당 ID을 부여하고 tagged에 추가
    tagged.append(
        TaggedDocument(words = toks, tags = [f'DOC_{idx}'])
    )
tagged

[TaggedDocument(words=[], tags=['DOC_0']),
 TaggedDocument(words=['서리', '굶주리'], tags=['DOC_1']),
 TaggedDocument(words=[], tags=['DOC_2']),
 TaggedDocument(words=['우연히', '돌리', '사로잡히', '최근', '10년', '코미디 영화', '최고'], tags=['DOC_3']),
 TaggedDocument(words=['찜찜', '감독', '소유진', '죽이'], tags=['DOC_4']),
 TaggedDocument(words=['그냥', '최고', '드라마', '그냥'], tags=['DOC_5']),
 TaggedDocument(words=['컴퓨터', '전기세', '아깝'], tags=['DOC_6']),
 TaggedDocument(words=['위하', '영화'], tags=['DOC_7']),
 TaggedDocument(words=['소문나', '잔치', '호화', '캐스팅'], tags=['DOC_8']),
 TaggedDocument(words=['종교', '분위기', '물씬', '풍기', '최루', '가족', '영화'], tags=['DOC_9']),
 TaggedDocument(words=['진짜', '재미있', '실망', '도라에몽', '극장판', '처음', '이건', '진짜', '재미있'], tags=['DOC_10']),
 TaggedDocument(words=['한일', '월드컵', '시절', '영화', '어설프', '부끄럽'], tags=['DOC_11']),
 TaggedDocument(words=['재밌', '영화'], tags=['DOC_12']),
 TaggedDocument(words=[], tags=['DOC_13']),
 TaggedDocument(words=['에반게리온', '인가'], tags=['DOC_14']),
 TaggedDocument(words=['시간', '다행', 

In [198]:
model = Doc2Vec(
    documents=tagged,
    vector_size=200,
    window=5,
    min_count=2,
    dm=1,
    negative=5,
    epochs=50,
    seed=42
)

In [199]:
print(model.dv[0])

[ 3.9937291e-03  4.6814997e-03  3.9908858e-03 -3.8314480e-03
 -4.2419406e-03 -2.2635579e-03 -2.1408796e-03  2.3017400e-03
 -3.3316566e-03 -4.2269509e-03 -2.3112886e-03  4.8547448e-03
 -2.3290981e-03  7.4957073e-04  4.3959604e-03  2.1659744e-03
 -4.8574508e-04  4.6649235e-03 -2.9733211e-03  2.7635538e-03
  2.0161439e-03 -1.7593526e-03 -4.6652733e-04 -4.4027977e-03
  2.6354790e-05 -1.3268429e-03  4.9558799e-03 -2.3597134e-03
 -3.3783133e-03  5.6155684e-04 -3.9088447e-03 -3.9828168e-03
 -3.2779609e-03 -3.0482465e-03 -2.1740568e-03 -9.8177430e-04
 -1.2414980e-03  2.8834527e-03  3.4169352e-03 -2.2675032e-03
 -3.1716502e-03  3.9009453e-04 -2.2545273e-03  1.2917519e-05
  3.5882085e-03  1.9421834e-03 -2.7052737e-03 -6.4148841e-04
  9.1473463e-05 -4.0412582e-03  4.9308627e-03  4.1577681e-03
 -1.4912486e-05  2.9250013e-03  2.2532283e-03  4.3304618e-03
  1.8252111e-03 -4.6403361e-03 -1.2257821e-03  1.7938280e-03
 -2.8638006e-04  2.4509085e-03  1.7629319e-03  2.2126525e-03
  2.7716870e-03  2.88476

- Doc2Vec 객체를 생성하여 학습
    - 매개변수
        - vector_size = 200
        - window = 5
        - min_count = 2
        - dm = 1
        - negative = 5
        - seed = 42
        - epochs = 50
    - 학습시키는 데이터는 X_train
- X_train, X_test -> 문자열 데이터 -> infer_vector() 함수를 이용해서 임베딩
- 고전 머신러닝 분류 모델을 이용하여 임베딩된 데이터를 독립 변수로 Y의 데이터들을 종속 변수로 학습하여 예측
    - 정확도를 확인
    - LogisticRegression(max_iter=2000, random_state=42)
    - LinearSVC(random_state=42)
    - 두 개의 모델을 사용하여 정확도가 좋은 모델을 선택

In [200]:
import re, collections
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [201]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

In [202]:
X_train_vec = [model.infer_vector(doc) for doc in X_train]
X_test_vec = [model.infer_vector(doc) for doc in X_test]

In [203]:
log_model = LogisticRegression(max_iter=2000, random_state=42)
log_model.fit(X_train_vec, Y_train)
log_pred = log_model.predict(X_test_vec)
log_acc = accuracy_score(Y_test, log_pred)
log_acc

0.71

In [204]:
svc_model = LinearSVC(random_state=42)
svc_model.fit(X_train_vec, Y_train)
svc_pred = svc_model.predict(X_test_vec)
svc_acc = accuracy_score(Y_test, svc_pred)
svc_acc

0.724

In [206]:
len(model.dv)

4000

In [207]:
len(tagged)

4000

In [215]:
print(len(X_train_vec))
print(len(X_test_vec))
print(len(Y_train))
print(len(Y_test))

4000
1000
4000
1000
